# distributed-sampler-shard — worked example 3: Manually replicate the DistributedSampler sharding formula

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `distributed-sampler-shard`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`DistributedSampler` works by first generating a full shuffled permutation of all indices, padding it to be divisible by `world_size`, then slicing out every `world_size`-th element starting at each rank's offset. Replicating this formula by hand helps you understand exactly which dataset indices each rank will see and why `set_epoch` must precede the iteration.

## Worked solution

For `n_samples=7`, `world_size=3`, `seed=42`, `epoch=0`:

**Step 1.** Compute `num_samples_per_rank = ceil(7/3) = 3`. Total padded size: `3*3 = 9`.

**Step 2.** Generate a shuffled permutation of indices `0..6` using `torch.randperm(7, generator=g)` where `g` is seeded with `seed + epoch = 42`.

**Step 3.** Pad the permutation to length 9 by repeating from the start: `perm + perm[:2]`.

**Step 4.** Rank 0 gets indices `[0, 3, 6]` of the padded list (every 3rd, starting at 0). Rank 1 gets `[1, 4, 7]`. Rank 2 gets `[2, 5, 8]`.

This is exactly what `DistributedSampler` does internally, making the formula transparent.

In [ ]:
import torch
import math
from torch.utils.data import TensorDataset
from torch.utils.data.distributed import DistributedSampler

def manual_shard_indices(n_samples, world_size, rank, seed, epoch):
    """Replicate DistributedSampler's index selection formula."""
    num_samples = math.ceil(n_samples / world_size)
    total_size = num_samples * world_size

    g = torch.Generator()
    g.manual_seed(seed + epoch)
    perm = torch.randperm(n_samples, generator=g).tolist()

    # Pad to total_size by repeating from the front
    perm = perm + perm[:(total_size - n_samples)]
    assert len(perm) == total_size

    # Each rank takes every world_size-th index starting at rank
    shard = perm[rank:total_size:world_size]
    return shard

# Compare against DistributedSampler ground truth
torch.manual_seed(0)
n_samples, world_size, seed, epoch = 11, 4, 55, 2
dataset = TensorDataset(torch.arange(n_samples))

print('Manual vs DistributedSampler comparison:')
for rank in range(world_size):
    manual = manual_shard_indices(n_samples, world_size, rank, seed, epoch)
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank,
                                  shuffle=True, seed=seed)
    sampler.set_epoch(epoch)
    official = list(sampler)
    match = manual == official
    print(f'  Rank {rank}: manual={manual}, official={official}, match={match}')